# 일반 함수의 Span 기록

- 구분: 정답 예시
- 원본: [`03_observe_decorator.py`](03_observe_decorator.py)
- 연결 교안: [4-2-langfuse-observability-and-evaluation.pdf](../../../99_pdf/Chapter02/day02/4-2-langfuse-observability-and-evaluation.pdf)

원본 Python 코드는 수정 없이 아래의 한 코드 셀에 담았습니다. 설명과 실행 준비는 별도 셀입니다. **모두 실행** 대신 필요한 셀을 직접 선택하세요.


## 학습 목표

1. Trace와 Span의 관계 및 `@observe`의 범위를 설명한다.
2. 코드를 실행하기 전 입력, 출력, 외부 호출과 파일 변경 가능성을 설명한다.
3. CodeMate가 만든 결과를 원본과 교안에 비춰 검토한다.


## 코드를 읽는 순서

retrieve → format_context → rag_pipeline에서 데코레이터의 중첩 관계를 읽는다.

교안은 개념과 목표를 설명합니다. 코드에서 실제 구현된 부분과 TODO를 구분하세요.


## 실행 준비

저장소 안에서 VS Code로 열고 프로젝트 `.venv` 커널을 선택합니다. 의존성은 `uv sync`로 준비합니다. 사내망에서는 사내 LLM 환경을, 사외에서는 허용된 provider를 준비합니다. 자격증명 값은 셀에 적지 않습니다. 아래 준비 셀은 저장소 루트를 찾고 `__file__`을 원본 Python 파일 경로로 설정합니다.


In [ ]:
from pathlib import Path
import os
import sys
_start = Path.cwd().resolve()
_repo = next((p for p in (_start, *_start.parents) if (p / "pyproject.toml").is_file() and (p / "config" / "llm.py").is_file()), None)
if _repo is None:
    raise FileNotFoundError("저장소 안에서 노트북을 열어 주세요.")
os.chdir(_repo)
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
__file__ = str(_repo / 'solution/Chapter02/05-observability/03_observe_decorator.py')
print(f"실행 기준 폴더: {_repo}")


## 원본 코드 — 직접 실행

**조건과 부작용:** 검색 문서는 모의 데이터다. LLM 호출과 Langfuse 전송이 발생한다. 입력 문구에 비공개 자료를 넣지 않는다.

준비 셀을 먼저 실행하고 필요한 서비스·설정을 확인하세요. 다음 셀은 `manual-run`으로 표시했습니다. 원본 전체가 한 번에 실행되므로 API 요청, 파일 변경 또는 대기가 생길 수 있습니다.


In [ ]:
"""
@observe 데코레이터: LangChain 밖의 코드도 트레이싱하기

핵심 개념:
  - CallbackHandler는 LangChain 체인 내부만 추적한다.
  - @observe 데코레이터는 일반 Python 함수도 span으로 기록할 수 있다.
  - RAG의 검색 단계, 전처리 함수, 비즈니스 로직 등을 함께 추적할 때 유용하다.
  - 중첩 함수에 각각 @observe를 붙이면 부모-자식 트리 구조로 기록됨

실행:
  python solution/Chapter02/05-observability/03_observe_decorator.py
"""

from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langfuse import get_client, observe
from langfuse.langchain import CallbackHandler

from config.llm import get_llm

load_dotenv()

langfuse = get_client()
langfuse_handler = CallbackHandler()
llm = get_llm()


# @observe(): 이 함수의 호출이 Langfuse에 span으로 기록됨
# name 파라미터로 대시보드에 표시될 이름을 지정할 수 있음
@observe(name='retrieve-documents')
def retrieve(query: str) -> list[str]:
    # 실제로는 벡터스토어에서 검색하는 부분
    # 여기서는 시뮬레이션용 더미 데이터 반환
    return [
        f'[문서1] {query}에 관한 첫 번째 참고 자료입니다.',
        f'[문서2] {query}와 관련된 두 번째 참고 자료입니다.',
    ]


@observe(name='format-context')
def format_context(docs: list[str]) -> str:
    return '\n'.join(docs)


# @observe로 감싼 함수 안에서 LangChain 체인을 호출하면
# 체인 실행이 이 span의 자식으로 자동 연결됨
@observe(name='rag-pipeline')
def rag_pipeline(question: str) -> str:
    docs = retrieve(question)
    context = format_context(docs)

    prompt = ChatPromptTemplate.from_messages([
        ('system', '아래 컨텍스트를 바탕으로 질문에 답하세요.\n\n{context}'),
        ('human', '{question}'),
    ])
    chain = prompt | llm | StrOutputParser()

    # CallbackHandler를 여기서도 전달해야 LLM 호출이 같은 트레이스에 묶임
    return chain.invoke(
        {'context': context, 'question': question},
        config={'callbacks': [langfuse_handler]},
    )


print('=== @observe 데코레이터 트레이싱 ===')
questions = ['LangGraph란 무엇인가요?', 'RAG의 장점은?']
for q in questions:
    answer = rag_pipeline(q)
    print(f'Q: {q}')
    print(f'A: {answer}')
    print()

langfuse.flush()
print('✓ 대시보드에서 retrieve → format → LLM 호출 트리를 확인하세요.')
print('Tracing에서 확인')


## CodeMate에게 요청하고 검토하기

> `@observe`와 `CallbackHandler`가 각각 무엇을 기록하는지 설명하고 예상 Trace 트리를 그려 줘.

CodeMate가 제안한 수정의 근거를 읽고, 실행 가능한 환경에서 출력·Trace·생성 파일을 확인하세요.


## 확인 질문

1. 검색 문서는 실제 자료인가? LLM 호출이 같은 Trace에 묶이려면 무엇을 전달해야 하는가?
2. 실행 전 어떤 환경과 자료를 확인해야 하는가?
3. 결과가 기대와 다르면 어느 단계의 기록부터 볼 것인가?
